In [0]:
from pyspark.sql import SparkSession
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

spark = SparkSession.builder.getOrCreate()

df_features = spark.table("analytics_ml.int_features.features_propensity_3").toPandas()

df_clv = spark.table("analytics_ml.int_features.features_clv_proxy")[["customerid","clv_proxy"]].toPandas()

# Merge features
df = df_features.merge(df_clv, on="customerid", how="left")

feature_cols = [
    "total_orders_base",
    "avg_order_value",
    "days_since_last_purchase",
    "count_email_open",
    "count_email_click",
    "count_sms_click",
    "count_push_open",
    "clv_proxy"
]

X = df[feature_cols]
y = df["label"]

# Optional: check label distribution
print("Label distribution:")
print(y.value_counts())
print("Label ratio (%):")
print(y.value_counts(normalize=True) * 100)

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

y_pred_proba = model.predict_proba(X)[:,1]  # probability for class 1
auc = roc_auc_score(y, y_pred_proba)
print(f"Propensity model AUC: {auc:.3f}")

# -----------------------------
# Step 5: Compute model signature for Unity Catalog
# -----------------------------
y_pred = model.predict(X)  # use class prediction or y_pred_proba
signature = infer_signature(X, y_pred)

# -----------------------------
# Step 6: Log and register the model
# -----------------------------
with mlflow.start_run(run_name="propensity_model_v1"):
    # Log metric
    mlflow.log_metric("AUC", auc)
    
    # Log model type
    mlflow.log_param("model_type", "logistic_regression")
    
    # Log and register model
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="propensity_model",
        registered_model_name="analytics_ml.int_features.PropensityModel_v1",
        signature=signature,
        input_example=X.head(3)  # optional but recommended
    )

print("Model successfully logged and registered with Unity Catalog!")

